# Task 2: Restaurant Recommendation System

## Objective

The objective of this task is to build a restaurant recommendation system based on user preferences.

The system uses a content-based filtering approach to recommend restaurants according to the user's preferred cuisine and price range.

The main steps include data preprocessing, handling missing values, encoding categorical features, calculating similarity, generating recommendations, testing the system, and evaluating recommendation quality.

## 1. Import Required Libraries

The required Python libraries are imported for data manipulation, data analysis, visualization, feature encoding, and similarity calculation.

Pandas is used for handling the dataset, NumPy is used for numerical operations, Matplotlib and Seaborn are used for visualization, and Scikit-learn is used for calculating cosine similarity.

In [208]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics.pairwise import cosine_similarity

## 2. Load the Dataset

The restaurant dataset is loaded into a Pandas DataFrame.

The dataset contains information about restaurants, including restaurant names, cuisines, price ranges, ratings, locations, and other restaurant-related attributes.

In [209]:
df = pd.read_csv('Dataset.csv')

df.head()

,Restaurant ID,Restaurant Name,Country Code,City,Address,Locality,Locality Verbose,Longitude,Latitude,Cuisines,...,Currency,Has Table booking,Has Online delivery,Is delivering now,Switch to order menu,Price range,Aggregate rating,Rating color,Rating text,Votes
0,6317637,Le Petit Souffle,162,Makati City,"Third Floor, Century City Mall, Kalayaan Avenu...","Century City Mall, Poblacion, Makati City","Century City Mall, Poblacion, Makati City, Mak...",121.027535,14.565443,"French, Japanese, Desserts",...,Botswana Pula(P),Yes,No,No,No,3,4.8,Dark Green,Excellent,314
1,6304287,Izakaya Kikufuji,162,Makati City,"Little Tokyo, 2277 Chino Roces Avenue, Legaspi...","Little Tokyo, Legaspi Village, Makati City","Little Tokyo, Legaspi Village, Makati City, Ma...",121.014101,14.553708,Japanese,...,Botswana Pula(P),Yes,No,No,No,3,4.5,Dark Green,Excellent,591
2,6300002,Heat - Edsa Shangri-La,162,Mandaluyong City,"Edsa Shangri-La, 1 Garden Way, Ortigas, Mandal...","Edsa Shangri-La, Ortigas, Mandaluyong City","Edsa Shangri-La, Ortigas, Mandaluyong City, Ma...",121.056831,14.581404,"Seafood, Asian, Filipino, Indian",...,Botswana Pula(P),Yes,No,No,No,4,4.4,Green,Very Good,270
3,6318506,Ooma,162,Mandaluyong City,"Third Floor, Mega Fashion Hall, SM Megamall, O...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.056475,14.585318,"Japanese, Sushi",...,Botswana Pula(P),No,No,No,No,4,4.9,Dark Green,Excellent,365
4,6314302,Sambo Kojin,162,Mandaluyong City,"Third Floor, Mega Atrium, SM Megamall, Ortigas...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.057508,14.584450,"Japanese, Korean",...,Botswana Pula(P),Yes,No,No,No,4,4.8,Dark Green,Excellent,229


## 3. Dataset Exploration

The dataset is explored to understand its structure and contents.

The number of rows and columns, column names, data types, and basic statistical information are examined. This helps identify the features that can be used for building the recommendation system.

In [210]:
print("Dataset Shape:", df.shape)

Dataset Shape: (9551, 21)


In [211]:
df.columns

Index(['Restaurant ID', 'Restaurant Name', 'Country Code', 'City', 'Address',
       'Locality', 'Locality Verbose', 'Longitude', 'Latitude', 'Cuisines',
       'Average Cost for two', 'Currency', 'Has Table booking',
       'Has Online delivery', 'Is delivering now', 'Switch to order menu',
       'Price range', 'Aggregate rating', 'Rating color', 'Rating text',
       'Votes'],
      dtype='object')

In [212]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9551 entries, 0 to 9550
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Restaurant ID         9551 non-null   int64  
 1   Restaurant Name       9551 non-null   object 
 2   Country Code          9551 non-null   int64  
 3   City                  9551 non-null   object 
 4   Address               9551 non-null   object 
 5   Locality              9551 non-null   object 
 6   Locality Verbose      9551 non-null   object 
 7   Longitude             9551 non-null   float64
 8   Latitude              9551 non-null   float64
 9   Cuisines              9542 non-null   object 
 10  Average Cost for two  9551 non-null   int64  
 11  Currency              9551 non-null   object 
 12  Has Table booking     9551 non-null   object 
 13  Has Online delivery   9551 non-null   object 
 14  Is delivering now     9551 non-null   object 
 15  Switch to order menu 

In [213]:
print(df.columns.tolist())


['Restaurant ID', 'Restaurant Name', 'Country Code', 'City', 'Address', 'Locality', 'Locality Verbose', 'Longitude', 'Latitude', 'Cuisines', 'Average Cost for two', 'Currency', 'Has Table booking', 'Has Online delivery', 'Is delivering now', 'Switch to order menu', 'Price range', 'Aggregate rating', 'Rating color', 'Rating text', 'Votes']


In [214]:
df.describe()

,Restaurant ID,Country Code,Longitude,Latitude,Average Cost for two,Price range,Aggregate rating,Votes
count,9.551000e+03,9551.000000,9551.000000,9551.000000,9551.000000,9551.000000,9551.000000,9551.000000
mean,9.051128e+06,18.365616,64.126574,25.854381,1199.210763,1.804837,2.666370,156.909748
std,8.791521e+06,56.750546,41.467058,11.007935,16121.183073,0.905609,1.516378,430.169145
min,5.300000e+01,1.000000,-157.948486,-41.330428,0.000000,1.000000,0.000000,0.000000
25%,3.019625e+05,1.000000,77.081343,28.478713,250.000000,1.000000,2.500000,5.000000
50%,6.004089e+06,1.000000,77.191964,28.570469,400.000000,2.000000,3.200000,31.000000
75%,1.835229e+07,1.000000,77.282006,28.642758,700.000000,2.000000,3.700000,131.000000
max,1.850065e+07,216.000000,174.832089,55.976980,800000.000000,4.000000,4.900000,10934.000000


In [215]:
df.tail()

,Restaurant ID,Restaurant Name,Country Code,City,Address,Locality,Locality Verbose,Longitude,Latitude,Cuisines,...,Currency,Has Table booking,Has Online delivery,Is delivering now,Switch to order menu,Price range,Aggregate rating,Rating color,Rating text,Votes
9546,5915730,Naml۱ Gurme,208,��stanbul,"Kemanke�� Karamustafa Pa��a Mahallesi, R۱ht۱m ...",Karak�_y,"Karak�_y, ��stanbul",28.977392,41.022793,Turkish,...,Turkish Lira(TL),No,No,No,No,3,4.1,Green,Very Good,788
9547,5908749,Ceviz A��ac۱,208,��stanbul,"Ko��uyolu Mahallesi, Muhittin ��st�_nda�� Cadd...",Ko��uyolu,"Ko��uyolu, ��stanbul",29.041297,41.009847,"World Cuisine, Patisserie, Cafe",...,Turkish Lira(TL),No,No,No,No,3,4.2,Green,Very Good,1034
9548,5915807,Huqqa,208,��stanbul,"Kuru�_e��me Mahallesi, Muallim Naci Caddesi, N...",Kuru�_e��me,"Kuru�_e��me, ��stanbul",29.034640,41.055817,"Italian, World Cuisine",...,Turkish Lira(TL),No,No,No,No,4,3.7,Yellow,Good,661
9549,5916112,A���k Kahve,208,��stanbul,"Kuru�_e��me Mahallesi, Muallim Naci Caddesi, N...",Kuru�_e��me,"Kuru�_e��me, ��stanbul",29.036019,41.057979,Restaurant Cafe,...,Turkish Lira(TL),No,No,No,No,4,4.0,Green,Very Good,901
9550,5927402,Walter's Coffee Roastery,208,��stanbul,"Cafea��a Mahallesi, Bademalt۱ Sokak, No 21/B, ...",Moda,"Moda, ��stanbul",29.026016,40.984776,Cafe,...,Turkish Lira(TL),No,No,No,No,2,4.0,Green,Very Good,591


## 4. Missing Value Analysis

Missing values are checked because incomplete information can affect the recommendation process.

The missing-value count and percentage are calculated to identify columns that require preprocessing.

In [216]:
missing_values = df.isnull().sum()

missing_percentage = (df.isnull().sum() / len(df)) * 100

missing_analysis = pd.DataFrame({
    'Missing Values': missing_values,
    'Missing Percentage': missing_percentage
})

missing_analysis

,Missing Values,Missing Percentage
Restaurant ID,0,0.000000
Restaurant Name,0,0.000000
Country Code,0,0.000000
City,0,0.000000
Address,0,0.000000
Locality,0,0.000000
Locality Verbose,0,0.000000
Longitude,0,0.000000
Latitude,0,0.000000
Cuisines,9,0.094231


## 5. Handling Missing Values

The Cuisines column contains a small number of missing values.

Since cuisine is an important feature for restaurant recommendation, the missing values are replaced with "Unknown" instead of removing the corresponding restaurant records.

## 5. Analyze Recommendation Criteria

For this recommendation system, the main user preferences are cuisine and price range.

Cuisine represents the type of food preferred by the user, while price range represents the user's preferred spending level.

These features are selected because they directly describe the user's restaurant preferences.

In [217]:
df['Cuisines'] = df['Cuisines'].fillna('Unknown')

df['Cuisines'].isnull().sum()

np.int64(0)

In [218]:
df['Cuisines'].nunique()

1826

In [219]:
df['Cuisines'].value_counts().head(20)

,count
Cuisines,
North Indian,936
"North Indian, Chinese",511
Fast Food,354
Chinese,354
"North Indian, Mughlai",334
Cafe,299
Bakery,218
"North Indian, Mughlai, Chinese",197
"Bakery, Desserts",170


In [220]:
df['Cuisines'].unique()[:20]

array(['French, Japanese, Desserts', 'Japanese',
       'Seafood, Asian, Filipino, Indian', 'Japanese, Sushi',
       'Japanese, Korean', 'Chinese', 'Asian, European',
       'Seafood, Filipino, Asian, European', 'European, Asian, Indian',
       'Filipino', 'Filipino, Mexican', 'American, Ice Cream, Desserts',
       'Korean', 'Cafe, American, Italian, Filipino', 'Italian, Pizza',
       'Cafe, Korean, Desserts', 'Cafe, Bakery, American, Italian',
       'Seafood, American, Mediterranean, Japanese',
       'American, Asian, Italian, Seafood', 'Fast Food, French'],
      dtype=object)

In [221]:
df['Price range'].value_counts().sort_index()

,count
Price range,
1,4444
2,3113
3,1408
4,586


In [222]:
df['Price range'].unique()

array([3, 4, 2, 1])

## 6. Duplicate Record Check

Duplicate records are checked to ensure that the same restaurant information is not unnecessarily repeated in the recommendation dataset.

In [223]:
recommendation_df = df[['Restaurant ID', 'Restaurant Name', 'Cuisines', 'Price range']].copy()

In [224]:
recommendation_df.head()

,Restaurant ID,Restaurant Name,Cuisines,Price range
0,6317637,Le Petit Souffle,"French, Japanese, Desserts",3
1,6304287,Izakaya Kikufuji,Japanese,3
2,6300002,Heat - Edsa Shangri-La,"Seafood, Asian, Filipino, Indian",4
3,6318506,Ooma,"Japanese, Sushi",4
4,6314302,Sambo Kojin,"Japanese, Korean",4


In [225]:
recommendation_df.isnull().sum()

,0
Restaurant ID,0
Restaurant Name,0
Cuisines,0
Price range,0


In [226]:
df.duplicated().sum()

np.int64(0)

## 7. Determine Recommendation Criteria

The recommendation criteria selected for this system are:

- Cuisine Preference
- Price Range

Cuisine preference represents the type of food preferred by the user, while price range represents the user's preferred spending level.

These two features are used to determine how closely each restaurant matches the user's preferences.

In [227]:
df['Cuisines'].nunique()

1826

In [228]:
df['Cuisines'].value_counts().head(20)

,count
Cuisines,
North Indian,936
"North Indian, Chinese",511
Fast Food,354
Chinese,354
"North Indian, Mughlai",334
Cafe,299
Bakery,218
"North Indian, Mughlai, Chinese",197
"Bakery, Desserts",170


In [229]:
df['Cuisines'].unique()[:20]

array(['French, Japanese, Desserts', 'Japanese',
       'Seafood, Asian, Filipino, Indian', 'Japanese, Sushi',
       'Japanese, Korean', 'Chinese', 'Asian, European',
       'Seafood, Filipino, Asian, European', 'European, Asian, Indian',
       'Filipino', 'Filipino, Mexican', 'American, Ice Cream, Desserts',
       'Korean', 'Cafe, American, Italian, Filipino', 'Italian, Pizza',
       'Cafe, Korean, Desserts', 'Cafe, Bakery, American, Italian',
       'Seafood, American, Mediterranean, Japanese',
       'American, Asian, Italian, Seafood', 'Fast Food, French'],
      dtype=object)

### Cuisine Analysis

The Cuisines column contains different cuisine types and combinations of cuisines.

Some restaurants offer more than one cuisine, such as North Indian and Chinese. Therefore, the cuisine information needs to be separated into individual cuisine features before calculating similarity.

In [230]:
df['Price range'].value_counts().sort_index()

,count
Price range,
1,4444
2,3113
3,1408
4,586


In [231]:
df['Price range'].unique()

array([3, 4, 2, 1])

### Price Range Analysis

The Price range column contains four categories: 1, 2, 3, and 4.

These values represent different price levels and are used as one of the user's recommendation preferences.

## 8. Prepare Recommendation Dataset

Only the columns required for the recommendation system are selected.

Restaurant ID and Restaurant Name are retained to identify the recommended restaurants.

Cuisines and Price range are selected as the main features used for generating recommendations.

In [232]:
recommendation_df = df[
    ['Restaurant ID', 'Restaurant Name', 'Cuisines', 'Price range']
].copy()

recommendation_df.head()

,Restaurant ID,Restaurant Name,Cuisines,Price range
0,6317637,Le Petit Souffle,"French, Japanese, Desserts",3
1,6304287,Izakaya Kikufuji,Japanese,3
2,6300002,Heat - Edsa Shangri-La,"Seafood, Asian, Filipino, Indian",4
3,6318506,Ooma,"Japanese, Sushi",4
4,6314302,Sambo Kojin,"Japanese, Korean",4


In [233]:
recommendation_df.isnull().sum()

,0
Restaurant ID,0
Restaurant Name,0
Cuisines,0
Price range,0


## 9. Feature Engineering

Feature engineering is performed to convert the restaurant information into a suitable numerical representation for the recommendation system.

The Cuisines column is cleaned before encoding because some restaurants contain multiple cuisine types.

The cuisine and price-range features are then converted into numerical features that can be compared using cosine similarity.

In [234]:
recommendation_df['Cuisines'] = recommendation_df['Cuisines'].str.strip()

## 10. Cuisine Feature Encoding

The Cuisines column may contain one or more cuisine types for each restaurant.

One-hot encoding is used to convert individual cuisines into binary features.

A value of 1 means that the restaurant offers the particular cuisine, while a value of 0 means that it does not.

In [235]:
cuisine_features = recommendation_df['Cuisines'].str.get_dummies(sep=', ')

cuisine_features.head()

,Afghani,African,American,Andhra,Arabian,Argentine,Armenian,Asian,Asian Fusion,Assamese,...,Tex-Mex,Thai,Tibetan,Turkish,Turkish Pizza,Unknown,Vegetarian,Vietnamese,Western,World Cuisine
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## 11. Price Range Encoding

The Price range column contains four categories: 1, 2, 3, and 4.

One-hot encoding is used to convert these categories into separate binary features: Price_1, Price_2, Price_3, and Price_4.

This allows the user's price preference to be compared with the price range of each restaurant during similarity calculation.

In [236]:
price_features = pd.get_dummies(
    recommendation_df['Price range'],
    prefix='Price'
)

price_features.head()

,Price_1,Price_2,Price_3,Price_4
0,False,False,True,False
1,False,False,True,False
2,False,False,False,True
3,False,False,False,True
4,False,False,False,True


## 12. Create the Feature Matrix

The cuisine features and price-range features are combined into a single feature matrix.

This matrix provides a numerical representation of each restaurant based on its cuisine and price range.

The feature matrix will be used as the input for calculating similarity.

In [237]:
feature_matrix = pd.concat(
    [cuisine_features, price_features],
    axis=1
)

In [238]:
feature_matrix.shape

(9551, 150)

## 13. Calculate Cosine Similarity

Cosine similarity is used to measure the similarity between the user's preference vector and restaurant feature vectors.

A similarity score closer to 1 indicates a stronger match, while a score closer to 0 indicates a weaker match.

The similarity scores are used to rank restaurants according to how closely they match the user's preferences.

In [239]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(feature_matrix)

similarity_matrix.shape

(9551, 9551)

## 14. Build the Content-Based Recommendation System

A recommendation function is created to accept the user's cuisine and price-range preferences.

The user's preferences are converted into the same feature representation used for the restaurants.

Cosine similarity is then calculated between the user preference vector and all restaurant feature vectors.

The restaurants are ranked according to their similarity scores, and the top matching restaurants are returned.

In [240]:
def recommend_restaurants(cuisine, price_range, top_n=5):

    # Create user preference vector
    user_features = pd.DataFrame(
        0,
        index=[0],
        columns=feature_matrix.columns
    )

    # Set cuisine preference
    if cuisine in user_features.columns:
        user_features[cuisine] = 1

    # Set price preference
    price_column = f'Price_{price_range}'

    if price_column in user_features.columns:
        user_features[price_column] = 1

    # Calculate similarity
    scores = cosine_similarity(
        user_features,
        feature_matrix
    )[0]

    # Get top restaurants
    top_indices = scores.argsort()[::-1][:top_n]

    # Create recommendation result
    recommendations = recommendation_df.iloc[top_indices].copy()

    # Add similarity score
    recommendations['Similarity Score'] = scores[top_indices]

    return recommendations

## 15. Test 1: Italian Cuisine and Price Range 2

The first test uses Italian cuisine and price range 2 as the user's preferences.

The system is asked to return the top 5 restaurants that best match these preferences.

In [241]:
recommendations = recommend_restaurants(
    cuisine='Italian',
    price_range=2,
    top_n=5
)

recommendations

,Restaurant ID,Restaurant Name,Cuisines,Price range,Similarity Score
233,17334965,Trattoria Tiramisu,Italian,2,1.0
532,17678097,Mom & Dad's Italian Restaurant,Italian,2,1.0
274,17342799,Vinny Vanucchi's,Italian,2,1.0
494,17621946,Trattoria Fresco,Italian,2,1.0
1106,18471318,Da Pizza Zone,Italian,2,1.0


## 16. Test 2: Chinese Cuisine and Price Range 3

The second test uses Chinese cuisine and price range 3.

This test checks whether the recommendation system can provide relevant recommendations for a different combination of cuisine and price preference.

In [242]:
recommendations_2 = recommend_restaurants(
    cuisine='Chinese',
    price_range=3,
    top_n=5
)

recommendations_2

,Restaurant ID,Restaurant Name,Cuisines,Price range,Similarity Score
5,18189371,Din Tai Fung,Chinese,3,1.0
1843,4133,Chin Chin,Chinese,3,1.0
9497,5800557,Chinese Dragon Cafe,Chinese,3,1.0
9200,3800052,Golden Dragon,Chinese,3,1.0
752,2600250,Chi Kitchen,Chinese,3,1.0


## 17. Test 3: Indian Cuisine and Price Range 1

The third test uses Indian cuisine and price range 1.

This provides another example of how the recommendation system responds to different user preferences.

In [243]:
recommendations_3 = recommend_restaurants(
    cuisine='Indian',
    price_range=1,
    top_n=5
)

recommendations_3

,Restaurant ID,Restaurant Name,Cuisines,Price range,Similarity Score
541,17697444,Masala Grill & Coffee House,"Indian, Middle Eastern",1,0.816497
3459,18370535,Meatwale.com,"North Indian, Indian",1,0.816497
2364,2300497,Atmosphere Grill Cafe Sheesha,"Indian, Chinese, Continental",1,0.707107
5824,18393213,Bhimsain's Bengali Sweet House,"Mithai, North Indian, Street Food, Chinese, So...",1,0.534522
984,8308,Sohan Sweets & Namkeen,Mithai,1,0.500000


## 18. Recommendation Quality Evaluation

The quality of the recommendations is evaluated by checking whether the recommended restaurants match the user's preferred cuisine and price range.

Three measures are used:

- Cuisine Match Rate: percentage of recommended restaurants matching the requested cuisine.
- Price Range Match Rate: percentage of recommended restaurants matching the requested price range.
- Average Similarity Score: average similarity score of the recommended restaurants.

These measures help evaluate how well the recommendation system satisfies the user's preferences.

In [244]:
cuisine_match = recommendations_3['Cuisines'].str.contains(
    'Indian',
    case=False,
    na=False
)

price_match = recommendations_3['Price range'] == 1

print("Cuisine Match Rate:", cuisine_match.mean() * 100, "%")
print("Price Range Match Rate:", price_match.mean() * 100, "%")
print("Average Similarity Score:", recommendations_3['Similarity Score'].mean())

Cuisine Match Rate: 80.0 %
Price Range Match Rate: 100.0 %
Average Similarity Score: 0.6749244853733696


## 19. Overall Evaluation Across All Tests

The results from all three test cases are combined to provide an overall evaluation of the recommendation system.

The cuisine match rate, price-range match rate, and average similarity score are compared across the three different user preference combinations.

This helps determine how consistently the recommendation system performs for different user preferences.

In [245]:
test_results = pd.DataFrame({
    'Test': ['Test 1', 'Test 2', 'Test 3'],
    'Cuisine': ['Italian', 'Chinese', 'Indian'],
    'Price Range': [2, 3, 1],

    'Cuisine Match Rate (%)': [
        recommendations['Cuisines'].str.contains(
            'Italian',
            case=False,
            na=False
        ).mean() * 100,

        recommendations_2['Cuisines'].str.contains(
            'Chinese',
            case=False,
            na=False
        ).mean() * 100,

        recommendations_3['Cuisines'].str.contains(
            'Indian',
            case=False,
            na=False
        ).mean() * 100
    ],

    'Price Match Rate (%)': [
        (recommendations['Price range'] == 2).mean() * 100,
        (recommendations_2['Price range'] == 3).mean() * 100,
        (recommendations_3['Price range'] == 1).mean() * 100
    ],

    'Average Similarity': [
        recommendations['Similarity Score'].mean(),
        recommendations_2['Similarity Score'].mean(),
        recommendations_3['Similarity Score'].mean()
    ]
})

test_results

,Test,Cuisine,Price Range,Cuisine Match Rate (%),Price Match Rate (%),Average Similarity
0,Test 1,Italian,2,100.0,100.0,1.000000
1,Test 2,Chinese,3,100.0,100.0,1.000000
2,Test 3,Indian,1,80.0,100.0,0.674924


## 20. Overall Recommendation Performance

The overall performance of the recommendation system is calculated by taking the average results across all three test cases.

The overall cuisine match rate, price-range match rate, and average similarity score provide a summary of the system's performance across different user preferences.

In [246]:
overall_cuisine_match = test_results['Cuisine Match Rate (%)'].mean()
overall_price_match = test_results['Price Match Rate (%)'].mean()
overall_similarity = test_results['Average Similarity'].mean()

print(
    "Overall Cuisine Match Rate:",
    round(overall_cuisine_match, 2),
    "%"
)

print(
    "Overall Price Match Rate:",
    round(overall_price_match, 2),
    "%"
)

print(
    "Overall Average Similarity:",
    round(overall_similarity, 4)
)

Overall Cuisine Match Rate: 93.33 %
Overall Price Match Rate: 100.0 %
Overall Average Similarity: 0.8916


# 21. Conclusion

The content-based restaurant recommendation system was successfully developed using cuisine and price-range preferences.

The Cuisines feature was converted into individual binary features, and the Price range feature was one-hot encoded. These features were combined into a feature matrix, and cosine similarity was used to measure how closely restaurants matched the user's preferences.

The system was tested using three different preference combinations:

- Italian cuisine with Price Range 2
- Chinese cuisine with Price Range 3
- Indian cuisine with Price Range 1

The overall evaluation achieved a 93.33% cuisine match rate and a 100% price-range match rate, with an overall average similarity score of 0.8916.

## Limitations

1. The current recommendation system mainly uses cuisine and price range.
2. Restaurants offering multiple cuisines can affect the cuisine match rate.
3. The recommendation system could be improved by including additional restaurant features such as location, ratings, votes, online delivery, and table booking.

Overall, the system demonstrates a content-based approach for recommending restaurants according to user preferences.